# Line Fitting — Watch a Model Learn From Data
**Demystifying AI — Industry Event, May 2026**

Welcome — your first hands-on moment of the session.

## First: what is this thing?

If you've never opened a notebook before, two minutes of orientation:

**What you're looking at is a *Jupyter notebook*, opened in *Google Colab*.** A notebook is a document that mixes blocks of writing (like this one) with blocks of code. The code blocks have a little ▶ button — clicking it runs that code on Google's computer and shows the result right below. Think of it as a lab notebook for software: code, results, and notes interleaved on one page.

**Google Colab** is Google's free service for running notebooks in the browser. You don't install anything. You don't need to know Python. Google hands every attendee a temporary virtual computer to run the demo on. Close the tab when you're done; nothing to clean up.

**What you're about to do:** click `Runtime → Run all` once. Two pictures will render — eight panels showing a line gradually settling onto its data, then four panels showing that same line getting yanked around by a handful of bad data points. Total run time: under 30 seconds.

**You do not need to read or understand the code.** Every block is commented for the curious, but the demo is in the pictures.

> **No GPU needed for this one.** Default Colab runtime (CPU) is fine — save your GPU allocation for the nanoGPT demo later in the session.

## What's in this notebook

- **Setup** — imports and a few knobs
- **Helpers** — three small functions: build fake data, fit a line, draw a plot
- **Phase 1** — watch a line stabilize as data grows from 2 points to 1,000
- **Phase 2** — inject 3 anomalies into a clean dataset; compare two fitting algorithms
- **Recap** — what this means and where the talk goes next


In [ ]:
# -------------------------------------------------------------------
# Setup — imports and the knobs that control the demo
# -------------------------------------------------------------------

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, HuberRegressor


# === The "ground truth" line ===
# We KNOW the real relationship is y = 2.5x + 4. We're going to generate
# noisy data around that line, then see how well different fitting
# algorithms recover the true slope (m) and intercept (b).
# This is the simplest possible version of "training a model on data."
TRUE_M = 2.5      # true slope
TRUE_B = 4.0      # true y-intercept
NOISE_STD = 3.0   # how noisy each data point is around the true line

# Seed the random number generator so everyone in the room sees the
# SAME data on their screen (otherwise each laptop would draw different
# points and the narration wouldn't match).
RNG = np.random.default_rng(42)


In [ ]:
# -------------------------------------------------------------------
# Three small helper functions — make data, fit a line, draw a plot
# -------------------------------------------------------------------


def make_data(n, noise_std=NOISE_STD):
    """Generate n points scattered around the true line y = TRUE_M * x + TRUE_B."""
    x = np.linspace(0, 20, n)              # n evenly-spaced x values from 0 to 20
    noise = RNG.normal(0, noise_std, n)    # Gaussian noise added to each y
    y = TRUE_M * x + TRUE_B + noise
    return x, y


def fit_ols(x, y):
    """Fit a line using ORDINARY LEAST SQUARES — the textbook regression.

    OLS = "find the m and b that minimize the sum of SQUARED errors."
    The squaring is the key detail: one big error counts more than many
    small ones. That's why OLS is sensitive to outliers (Phase 2 shows this).
    """
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def fit_huber(x, y):
    """Fit a line using HUBER REGRESSION — a "robust" alternative to OLS.

    Huber treats small errors like OLS does (squared), but treats LARGE
    errors LINEARLY instead of squaring them. That one change makes it
    much harder for a handful of bad points to drag the line off course.
    Used in any production setting where you expect dirty data.
    """
    model = HuberRegressor().fit(x.reshape(-1, 1), y)
    return float(model.coef_[0]), float(model.intercept_)


def draw(ax, x, y, m, b, title, extra_line=None, y_limits=None):
    """Draw one panel: scatter the points, plot the fit, plot the truth."""
    ax.clear()
    ax.scatter(x, y, s=30, alpha=0.7, label=f"data (n={len(x)})")

    # The line our model fit (red, solid)
    xs = np.array([0, 20])
    ax.plot(xs, m * xs + b, "r-", lw=2.2,
            label=f"OLS fit: y = {m:.2f}x + {b:.2f}")

    # Optional second line (used in Phase 2 to show Huber alongside OLS)
    if extra_line is not None:
        m2, b2, label2 = extra_line
        ax.plot(xs, m2 * xs + b2, "g--", lw=2.2, label=label2)

    # The true line we know we should be recovering (black dotted)
    ax.plot(xs, TRUE_M * xs + TRUE_B, "k:", lw=1, alpha=0.6,
            label=f"truth: y = {TRUE_M}x + {TRUE_B}")

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 20)
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)


## Phase 1 — Watch a line stabilize as data grows

We start with just **2 noisy data points** and fit a line through them. Then we add more points — 3, 5, 10, 30, 100, 300, 1000 — and refit each time. The eight panels below show what happens.

**What to watch for:** with only 2 points, the line is determined entirely by them — it can be wildly off. As points accumulate, the line converges to the truth (the dotted black line in every panel). By N=100 the line has essentially stopped moving. After that, more data barely changes anything.

**That is training, in its simplest possible form.** A computer trying different m and b values, keeping the pair that minimizes error.


In [ ]:
# -------------------------------------------------------------------
# Phase 1 — Fit a line at N = 2, 3, 5, 10, 30, 100, 300, 1000
# Render all 8 fits side-by-side so the convergence story is visible.
# -------------------------------------------------------------------

ns = [2, 3, 5, 10, 30, 100, 300, 1000]
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Phase 1 — the fit stabilizes as more data arrives",
             fontsize=15, fontweight="bold")

for ax, n in zip(axes.flat, ns):
    x, y = make_data(n)
    m, b = fit_ols(x, y)
    draw(ax, x, y, m, b,
         f"N = {n}   (fit: m={m:.2f}, b={b:.2f})",
         y_limits=(-10, 70))

plt.tight_layout()
plt.show()


**What you should be seeing above:**

- At **N=2**, the line passes exactly through both points — no error to minimize.
- At **N=3**, the line has to compromise for the first time. Already pretty good.
- At **N=10**, the line is close to the truth (dotted black).
- At **N=100**, the line has essentially stopped moving. **This is what "training converges" means.**
- At **N=1000**, same line as N=100. More data has diminishing returns once the signal is captured.

> *"You just watched the entire machine-learning loop. The computer tried values of m and b, measured how wrong they were, and kept the best pair. It just happened to be invisible because the function is this simple."*


## Phase 2 — Bad data tilts the line

Real data is messy. Sensor glitches, typos, fat-fingered entries. What happens when a clean 100-point dataset picks up a few wildly-wrong points?

Below, we start with the clean dataset and progressively inject 1, 2, then 3 anomalies. We fit the same data two different ways:

- **OLS** (red, solid) — what most regressions use by default. Sensitive to outliers.
- **Huber** (green, dashed) — a "robust" alternative that down-weights outliers instead of squaring them.

**Watch the red line tilt. Watch the green line stay put.**


In [ ]:
# -------------------------------------------------------------------
# Phase 2 — Inject anomalies one at a time, compare OLS vs Huber
# -------------------------------------------------------------------

# Start with a clean 100-point dataset
x_clean, y_clean = make_data(100)

# Three anomalies — chosen to be obviously, comically wrong
# (a real-world sensor glitch or data-entry typo would look similar)
anomalies_x = np.array([3.0, 8.0, 14.0])
anomalies_y = np.array([95.0, -35.0, 120.0])

fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
fig.suptitle("Phase 2 — three anomalies injected one at a time",
             fontsize=15, fontweight="bold")

for ax, k in zip(axes, range(4)):
    # k = number of anomalies injected so far (0, 1, 2, 3)
    xk = np.concatenate([x_clean, anomalies_x[:k]])
    yk = np.concatenate([y_clean, anomalies_y[:k]])

    # Fit OLS to the (clean + k anomalies) data
    m_ols, b_ols = fit_ols(xk, yk)

    # Also fit Huber once there's at least one anomaly to ignore
    extra = None
    if k >= 1:
        m_h, b_h = fit_huber(xk, yk)
        extra = (m_h, b_h, f"Huber (robust): y = {m_h:.2f}x + {b_h:.2f}")

    label = "anomaly" if k == 1 else "anomalies"
    draw(ax, xk, yk, m_ols, b_ols,
         f"{k} {label} added (N = {len(xk)})",
         extra_line=extra, y_limits=(-50, 140))

plt.tight_layout()
plt.show()


**What you should be seeing above:**

- **0 anomalies:** OLS sits right on top of the truth. No drama.
- **1 anomaly:** ONE bad point pulled the red OLS line off course. Huber barely moved.
- **2 anomalies:** OLS now visibly tilted. Huber still on the truth.
- **3 anomalies:** OLS is meaningfully wrong. Huber is still right.

> *"The algorithm determines how your model fails when reality is messy. When a vendor shows you a benchmark, they've shown you the red line on clean data. The real question — the one that predicts whether their model survives your production environment — is what happens when you hand it your messy data."*

---

## Where this goes next

Everything you just watched was **two numbers** — m and b — being adjusted to fit data. GPT-4 has **1.76 trillion** of those numbers. The adjustment procedure is the same. The only things that change are the function's shape, the parameter count, and how much compute you throw at it.

Keep that image in your head for the rest of the talk — because every architecture we're about to discuss (tokenization, embeddings, attention, transformers, the nanoGPT demo you'll run later) is a more elaborate version of this exact picture.
